# 🧠 Fine tuning Notes — DistilBERT Fine-Tuning
 
---
 
## 1. Key Libraries
 
| Library | Purpose |
|---|---|
| `torch` | Core deep learning math engine (PyTorch) |
| `datasets` | Load pre-built datasets from Hugging Face Hub |
| `transformers` | Pre-trained models, tokenizers, training utilities |
| `sklearn` | Utility metrics like `accuracy_score` |
 
---
 
## 2. Tokenizer
 
### What is it?
- Converts raw text → numbers that the neural network can process
- Neural networks are **pure math** — they cannot read English, only numbers
### How it works:
```
"The food was great"  →  [101, 1996, 2833, 2001, 2307, 102]
```
 
### Key points:
- Every pre-trained model has its **own matching tokenizer**
- Using the wrong tokenizer = wrong numbers = broken model
- `AutoTokenizer` automatically loads the correct tokenizer for any model
- Without `AutoTokenizer`, you'd have to manually research which tokenizer each model uses
### Tokenizer arguments:
| Argument | Meaning |
|---|---|
| `padding="max_length"` | Short sentences padded with zeros to reach max_length |
| `truncation=True` | Long sentences cut off at max_length |
| `max_length=128` | Every input will be exactly 128 tokens |
 
---
 
## 3. Model Architecture
 
### Two parts of DistilBERT:
 
```
Raw Text
    ↓
┌─────────────────┐
│   DistilBERT    │  ← Body: pre-trained, understands language
│  (768 numbers)  │    outputs rich vector representation
└─────────────────┘
    ↓
┌─────────────────┐
│  Linear Layer   │  ← Classification Head: randomly initialized
│   768 → 2       │    newly added for your specific task
└─────────────────┘
    ↓
[score_neg, score_pos]
```
 
### Classification Head:
- A simple linear layer added on top of the pre-trained body
- Maps model output (768 numbers) → your number of classes (2 for sentiment)
- **Randomly initialized** at the start — learns during fine-tuning
- If you had a 5-star task, the head would be `768 → 5` instead
---
 
## 4. Fine-Tuning
 
### Definition:
Fine-tuning = taking a pre-trained model and **gently adapting** its weights to a new specific task using a **very small learning rate**.
 
### Two key ingredients:
1. **Small learning rate** (e.g. `2e-5`) — preserves pre-trained knowledge
2. **New dataset** — guides the weights toward your specific task
### Why small learning rate matters:
 
| Learning Rate | Effect |
|---|---|
| Large (`0.1`) | Weights change dramatically → destroys pre-trained knowledge |
| Tiny (`2e-5`) | Weights change microscopically → preserves knowledge, specializes slowly |
 
### Fine-tuning vs Training from scratch:
 
| | Fine-Tuning | Training from Scratch |
|---|---|---|
| Starting weights | Pre-trained (meaningful) | Random (meaningless) |
| Learning rate | Very small (`1e-5` to `5e-5`) | Large (`0.001` to `0.1`) |
| Epochs needed | Few (2–5) | Many (50–100+) |
| Data needed | Small dataset works | Needs huge dataset |
| Goal | Adapt existing knowledge | Learn everything from zero |
 
### Learning rate rule of thumb:
- Fine-tuning a pre-trained model → `1e-5` to `5e-5`
- Training from scratch → `0.001` to `0.1`
---
 
## 5. Dataset
 
### Yelp Polarity:
- Binary sentiment classification: **0 = negative**, **1 = positive**
- Pre-split into `train` and `test` by Hugging Face — they never overlap
### Why no data leakage:
```python
dataset["train"]  # completely separate pool of reviews
dataset["test"]   # completely separate pool of reviews
```
- `.shuffle()` only randomizes within each split — it does NOT mix train and test
### The 3 columns used:
| Column | Meaning |
|---|---|
| `input_ids` | Tokenized text as numbers |
| `attention_mask` | 1 = real token, 0 = padding (model ignores padding) |
| `label` | 0 (negative) or 1 (positive) |
 
---
 
## 6. Training Pipeline
 
```
Raw text reviews
      ↓  tokenizer
Numbers + attention masks
      ↓  DistilBERT body
768-dimensional embeddings
      ↓  classification head
[score_negative, score_positive]
      ↓  argmax
Prediction (0 or 1)
      ↓  compare to label
Loss → backprop → update weights
```
 
### TrainingArguments key settings:
| Argument | Value | Meaning |
|---|---|---|
| `learning_rate` | `2e-5` | Tiny — preserves pre-trained knowledge |
| `num_train_epochs` | `2` | Pass through all data 2 times |
| `per_device_train_batch_size` | `8` | 8 examples processed per step |
| `weight_decay` | `0.01` | Regularization — prevents overfitting |
| `eval_strategy` | `"epoch"` | Evaluate after each epoch |
 
---
 
## 7. Key Interview Talking Points
 
- **Why use a pre-trained model?** It already understands language — you don't need to teach it grammar, vocabulary, context from scratch. Saves time, data, and compute.
- **What is a classification head?** A simple linear layer added on top of the pre-trained body to map its output to your specific number of classes.
- **Why is the tokenizer separate from the model?** Different tasks can share the same tokenizer. Tokenization runs on CPU, model runs on GPU. Tokenizer has no neural network weights — it's just a lookup table.
- **What makes fine-tuning different from training from scratch?** The learning rate. Fine-tuning uses a tiny learning rate to gently nudge pre-existing weights. Training from scratch uses a large learning rate because there's no prior knowledge to preserve.
- **What is data leakage?** When your test data is seen during training, giving falsely high accuracy. Avoided here because train and test come from pre-separated dataset splits.
 